# Digit recognizer
Programa creado para clasificar imágenes 28x28 de dígitos según su etiqueta 0-9 con algoritmos de Inteligencia Artificial

### IDEAS
- Darle pesos (con ML?) a cada imagen del dataset en la creación de los números ideales, ya que no todas son igual de representativas
- Probar con redes neuronales (NNs)

# NOTAS
* Tambíen se pueden invertir los errores multiplicando por -1
* Cambiar métrica de evaluación para clasificación multiclase

# Análisis del dataset

In [49]:
"""
IMPORTANTE:
En Github no se encuentran los archivos de 'train.csv' y 'test.csv' (pesan mucho)
Sin embargo, si que tenemos una carpeta ZIP con esos archivos. Para ejecutar el código,
se deben extraer primero (dentro de la carpeta)
"""

import pandas as pd

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [50]:
train.describe()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
count,42000.000000,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,...,42000.000000,42000.000000,42000.000000,42000.00000,42000.000000,42000.000000,42000.0,42000.0,42000.0,42000.0
mean,4.456643,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.219286,0.117095,0.059024,0.02019,0.017238,0.002857,0.0,0.0,0.0,0.0
std,2.887730,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6.312890,4.633819,3.274488,1.75987,1.894498,0.414264,0.0,0.0,0.0,0.0
min,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
25%,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
50%,4.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
75%,7.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
max,9.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,254.000000,254.000000,253.000000,253.00000,254.000000,62.000000,0.0,0.0,0.0,0.0


### Creación de los conjuntos 'train' y 'test'

In [ ]:
# Para reducir los tiempos de espera, se reduce el dataframe de train
# cantidad_datos = int(input("Cantidad de datos para entrenar")
# train = train[0:7000]
print(f"Cantidad de dígitos en train:  {len(train)}")
# porcentaje = float(input("Porcentaje de train sobre el total"))


porcentaje = 0.8 


#Split de los dos sets
entrenamiento = train[: int(len(train) * porcentaje)]
test = train[: int(len(train) * (1-porcentaje))]

# X_entrenamiento es el % de todos los dígitos sin la columna con la etiqueta 
X_entrenamiento = entrenamiento[: int(len(train) * porcentaje)]
X_entrenamiento.drop(["label"], axis=1)

# Y para entrenamiento es la columna con la etiqueta
Y_entrenamiento = entrenamiento["label"]

print("Información X de entrenamiento: \n")
X_entrenamiento.info()
print("Información Y de entrenamiento: \n")
Y_entrenamiento.info()

# X_test es el 1-% de todos los dígitos sin la columna con la etiqueta 
X_test = test.drop(["label"], axis=1)

# Y_test es la columan de la etiqueta
Y_test = test["label"]

print("Información X de test: \n")
X_test.info()
print("Información Y de test: \n")
Y_test.info()

print(f"Split hecho correctamente : {len(X_entrenamiento) + len(X_test) == len(train)}")


Cantidad de dígitos en train:  42000
Información X de entrenamiento: 

<class 'pandas.DataFrame'>
RangeIndex: 33600 entries, 0 to 33599
Columns: 785 entries, label to pixel783
dtypes: int64(785)
memory usage: 201.2 MB
Información Y de entrenamiento: 

<class 'pandas.Series'>
RangeIndex: 33600 entries, 0 to 33599
Series name: label
Non-Null Count  Dtype
--------------  -----
33600 non-null  int64
dtypes: int64(1)
memory usage: 262.6 KB
Información X de test: 

<class 'pandas.DataFrame'>
RangeIndex: 8399 entries, 0 to 8398
Columns: 784 entries, pixel0 to pixel783
dtypes: int64(784)
memory usage: 50.2 MB
Información Y de test: 

<class 'pandas.Series'>
RangeIndex: 8399 entries, 0 to 8398
Series name: label
Non-Null Count  Dtype
--------------  -----
8399 non-null   int64
dtypes: int64(1)
memory usage: 65.7 KB


In [ ]:
prueba = train[: int(len(train) * porcentaje)]
prueba

In [ ]:
train_nulls = train.isnull().sum().values
test_nulls = test.isnull().sum().values

print(f"Número de valores nulos en train: {train_nulls.sum()}")
print(f"Número de valores nulos en test: {test_nulls.sum()}")

Vemos en la documentación (https://www.kaggle.com/competitions/digit-recognizer) que las imágenes tienen 28x28 píxeles en escala de grises, con valores entre 0-255

Primero, vamos a normalizar las columnas para que tengan valores entre 0 y 1. Luego, crearçmos una función que muestre por pantalla una imagen específica del dataset para visualizarla mejor

In [ ]:
# Dividimos cada columna entre 255
for col in train.columns:
    if col != "label":
        train[col] /= 255

for col in test.columns:
    if col != "label":
        test[col] /= 255

# Comprobamos que haya funcionado
# train.describe()

In [ ]:
# Como parece que vamos a usar mucho este código, añado una función que convierta una fila de los datos en un array 2D
def crearMatriz(df, i):
    img = df.iloc[i]

    # Quitamos la columna 'label' si existe
    title = f"Image #{i}"
    label = None
    if "label" in img.index:
        title += f" (Number {int(img["label"])})"
        label = int(img["label"])
        # Convertimos el data series de pandas a un array de numpy
        # y eliminamos el valor "label" (ubicado en la primera posición)
        img = img.values[1:]
    else:
        img = img.values
        

    # Ahora, transformamos el array en un array 2D, 28x28
    img = img.reshape((28, 28))
    
    return img, label, title

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def mostrarImagenDF(df, i):
    """
    Mostrar una imágen a partir de una columna del DataFrame
    df: dataframe
    i: integer
    """
    img, label, title = crearMatriz(df, i)

    plt.imshow(img, cmap="gray", vmin=0, vmax=1)
    plt.title(label=title)
    plt.show()


def mostrarMatrizImagen(img, title=""):
    """
    Mostrar una imágen a partir de una matriz de píxeles
    img: Matriz de valores [0, 1]
    title: Texto para mostrar 
    """

    plt.imshow(img, cmap="gray", vmin=0, vmax=1)
    plt.title(label=title)
    plt.show()



Creamos una función para visualizar todos los datos de un dataset [simplemente para probar que números aparecen]

In [ ]:
from IPython.display import clear_output


def mostrarImgsDataframe(df, inicio, fin):
    # Activa modo interactivo
    plt.ion()

    for i in range(inicio, fin):
        # Usamos el código anterior para dibujar la matriz 2D
        img, label, title = crearMatriz(df, i)

        plt.imshow(img, cmap="gray", vmin=0, vmax=1)
        plt.title(label=title)
        plt.axis("off")

        # Refrescamos la ventana y esperamos N segundos
        plt.pause(0.02)

        # Borramos el output anterior
        clear_output(wait=True)
        plt.show()


# mostrarImgsDataframe(train, 0, 20)

# Creación de las matrices de 'números ideales'

Con el fin de evaluar a que label corresponde cada matriz, vamos a comprobar a que 'número ideal' se parece más dicha matriz. Estos 'números ideales' serán calculados como la media de los números de ese 'label'; es decir, se superponen los números uno encima de otro y se calcula cual es el 'número promedio' o 'número ideal'

In [ ]:
# Función para mostrar las imágenes de esos números ideales. La usaremos más adelante
def mostrarNumsIdeales(nums, figsize=(7, 7)):
    """
    nums: list de números ideales
    """

    fig, axs = plt.subplots(nrows=2, ncols=5, figsize=figsize)
    # Convertimos la matriz 2D de sublots en una lista 1D
    axs = axs.flatten()

    for i in range(len(nums)):
        img = nums[i]
        title = f"Número {i}"

        axs[i].set_title(title)
        axs[i].imshow(img, cmap="gray", vmin=0, vmax=1)
        axs[i].axis("off")

    plt.show()

In [ ]:
# --- Para el conjunto 'train', se crea un número ideal para cada label 0-9 ---

# Creamos unas matrices para guardar la suma/superposición de matrices
nums_ideales = []
for i in range(10):
    matriz = np.zeros((28, 28))
    nums_ideales.append(matriz)


for i in range(len(train)):
    vector_img, label, _ = crearMatriz(train, i)

    # Sumamos los valores de cada pixel a su matriz correspondiente
    nums_ideales[label] += vector_img

media_total_pixeles = 0

# Ahora, necesitamos normalizar los valores entre 0 y 1
for i in range(len(nums_ideales)):
    valor_maximo = nums_ideales[i].max()
    if valor_maximo == 0:
        raise ValueError("La matriz está vacía")

    # El valor máximo tendrá un valor de (1 / valor_maximo) * valor_maximo = 1
    # Y el mínimo de (1 / valor_maximo) * 0 = 0
    nums_ideales[i] = np.multiply(1 / valor_maximo, nums_ideales[i])

    # -- Suavizamos la nube borrosa incluyendo solo los datos por encima de un valor límite --
    limite = 0.5
    # Coeficiente de suavizado (0 elimina el píxel, 1 no hace ningún cambio)
    coef_suavizado = 0.75

    # for x in range(nums_ideales[i].shape[0]):
    #     for y in range(nums_ideales[i].shape[1]):
    #         # Si es menor que el límite, suavizamos su valor
    #         if nums_ideales[i][x][y] < limite:
    #             nums_ideales[i][x][y] *= coef_suavizado
    media_total_pixeles += nums_ideales[i].sum()

# media_total_pixeles /= len(nums_ideales)
# print("media", media_total_pixeles)
# # -- Queremos que todas las matrices tengan la misma cantidad de pixeles blancos --
# # La suma de todos los pixeles debe ser 'media_total_pixeles', por lo que debemos dividir cada
# # pixel por la diferencia (sum() / media_total_pixeles)
# for i in range(len(nums_ideales)):
#     suma = nums_ideales[i].sum()
#     factor_mult = media_total_pixeles / suma
#     nums_ideales[i] *= factor_mult

# # Comprobamos
# for i in range(len(nums_ideales)):
#     suma = nums_ideales[i].sum()
#     print(i, suma)

In [ ]:
# Podemos visualizar el resultado usando las funciones anteriores
mostrarNumsIdeales(nums_ideales)

# Comparación de los datos y los números ideales

Ahora, podemos ver si somos capaces de identificar una imagen por su similitud a un número ideal

Para ello, compararemos esa imagen con cada número ideal y elegimos la que sea más parecida (la que tenga menor error)

### Método 1
Diferencia entre matrices con pixeles con valores contínuos [0, 1]

In [ ]:
def predict(df, idx, mostrarErrores=True):
    img, label, title = crearMatriz(df, idx)

    # Guardamos la matriz de error de cada número ideal
    matrices_resta = []
    errores = []

    """
    * Para el caso discreto en el que un pixel pertenece a {0, 1}
    Llamamos p al valor del pixel y P al valor esperado (valor del píxel ideal)
    Queremos minimizar, así que el número que estamos buscando tendrá el menor valor
        - If (p == 0 and P == 0) or (p == 1 and P == 1):
    Es lo que buscamos, no sumamos nada
        - If (p == 0 and P == 1) or (p == 1 and P == 0):
    Añadimos 1 a la suma; ese pixel no se corresponde al ideal
    
    * Para el caso continuo en el que un pixel pertenece al intervalo [0, 1]
        - If (p == 0 and P == 0) or (p == 1 and P == 1):
    Es lo que buscamos, no sumamos nada
        - Else:
    # Tanto si el pixel debería ser más claro o más oscuro, añadimos ese error/diferencia
    # NOTA: Quizá se puede penalizar más a los pixeles que deberían ser oscuros y son claros (o viceversa)
    Añadimos abs(p - P) a la suma; ese pixel no se corresponde al ideal
    
    --> Al final, se deduce que se quiere calcular abs(p - P) para cada pixel, por lo que se puede operar
    la matriz entera usando métodos de Numpy para conseguir el mismo resultado
    """
    for i in range(len(nums_ideales)):

        matriz_differencia = np.abs(np.subtract(nums_ideales[i], img))
        err = np.sum(matriz_differencia)
        err = float(round(err, 3))  # Para mejorar la lectura de los datos

        matrices_resta.append(matriz_differencia)
        errores.append(err)

    # Normalizamos los errores para conseguir la probabilidad
    # El menor error tendrá la probabilidad mayor (de manera proporcional)
    # Primero, "invertimos" los valores para que el menor error tenga el mayor valor
    max_err = np.max(errores)
    for e in range(len(errores)):
        err = max_err - errores[e]
        errores[e] = float(round(err, 5))

    # Después calculamos la probabilidad normalizada de elegir cada número ideal
    vector_probabilidades = []
    for i in range(len(errores)):
        probabilidad = float(errores[i] / np.sum(errores))
        vector_probabilidades.append(round(probabilidad, 5))
    """
    Errores (abs):                [158.783, 131.972, 150.416, 153.856, 94.291, 153.081, 129.619, 132.346, 145.306, 116.825]
    Errores "invertidos":         [0.0, 26.811, 8.367, 4.927, 64.492, 5.702, 29.164, 26.437, 13.477, 41.958]
    Probabilidades normalizadas:  [0.0, 0.12113, 0.0378, 0.02226, 0.29138, 0.02576, 0.13176, 0.11944, 0.06089, 0.18957]
    """

    prediccion = vector_probabilidades.index(max(vector_probabilidades))


    if mostrarErrores:
        print(f"Errores:\n{errores}\tMínimo err: {min(errores)}")
        print(f"Probabilidades:\n{vector_probabilidades} ----> {prediccion}")
        print("sumatorio probs.", np.sum(vector_probabilidades))
        mostrarNumsIdeales(matrices_resta, figsize=(7, 5))

    # Devolvemos el valor del mínimo error; es decir, el índice de la matriz correspondiente
    return prediccion

In [ ]:
# Probamos a predecir todos los números de 'train'          !!!!!!!!!!!!!!!!!!!!!!!!! NO estamos probando en 'test' !!!!!!!!!
# Guardamos el total de aciertos para cada número
aciertos = [0] * 10
fallos = [0] * 10

for i in range(len(train)):
    prediccion = predict(train, i, mostrarErrores=False)
    target = int(train.iloc[i]["label"])

    # Sumamos uno si acierta
    if prediccion == target:
        aciertos[target] += 1
    else:
        fallos[target] += 1


In [ ]:
def calcular_resultados(aciertos, fallos):
    print("Aciertos:")
    print(aciertos)
    print("Fallos:")
    print(fallos)

    print("\nPorcentajes de acierto para cada clase ideal:")
    porcentajes = []
    for i in range(10):
        porcentaje = aciertos[i] / (aciertos[i] + fallos[i])
        porcentajes.append(round(porcentaje, 5))
    print(porcentajes)

    media = np.mean(porcentajes)
    print("\nPorcentaje medio de acierto: ", media)

    # --- Lo visualizamos todo como un DataFrame ---
    resultados = {
        "Aciertos": aciertos + [np.mean(aciertos)],
        "Fallos": fallos + [np.mean(fallos)],
        "Porcentaje": porcentajes + [media],
        "Número": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, "MEDIA"],
    }
    df_resultados = pd.DataFrame(resultados)
    # El índice para cada elemento pasa de ser 0, 1, 2... a ser los elementos de 'Número" (que en este caso son iguales)
    df_resultados.set_index("Número", inplace=True)
    
    return df_resultados

df_resultados1 = calcular_resultados(aciertos, fallos)
df_resultados1

### Método 2
Diferencia entre matrices con pixeles de valores discretos {0, 1}

In [ ]:
def predict2(df, idx, mostrarErrores=True):
    img, label, title = crearMatriz(df, idx)

    # Guardamos la matriz de error de cada número ideal
    matrices_resta = []
    errores = []

    """
    Mismo procedimiento (valor absoluto de la resta), pero primero "binarizamos" la matriz 'img'
    """
    # Valores entre 0 y 1
    img_discreta = img.copy()

    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            img_discreta[i][j] = round(img[i][j])

    for i in range(len(nums_ideales)):

        matriz_differencia = np.abs(np.subtract(nums_ideales[i], img_discreta))
        err = np.sum(matriz_differencia)
        err = float(round(err, 3))  # Para mejorar la lectura de los datos

        matrices_resta.append(matriz_differencia)
        errores.append(err)

    # Normalizamos los errores para conseguir la probabilidad
    # El menor error tendrá la probabilidad mayor (de manera proporcional)
    # Primero, "invertimos" los valores para que el menor error tenga el mayor valor
    max_err = np.max(errores)
    for e in range(len(errores)):
        err = max_err - errores[e]
        errores[e] = float(round(err, 5))

    # Después calculamos la probabilidad normalizada de elegir cada número ideal
    vector_probabilidades = []
    for i in range(len(errores)):
        probabilidad = float(errores[i] / np.sum(errores))
        vector_probabilidades.append(round(probabilidad, 5))

    prediccion = vector_probabilidades.index(max(vector_probabilidades))

    if mostrarErrores:
        print(f"Errores:\n{errores}\tMínimo err: {min(errores)}")
        print(f"Probabilidades:\n{vector_probabilidades} ----> {prediccion}")
        print("sumatorio probs.", np.sum(vector_probabilidades))
        mostrarNumsIdeales(matrices_resta, figsize=(7, 5))

    # Devolvemos el valor del mínimo error; es decir, el índice de la matriz correspondiente
    return prediccion

In [ ]:
# Probamos a predecir todos los números de 'train'          !!!!!!!!!!!!!!!!!!!!!!!!! NO estamos probando en 'test' !!!!!!!!!
# Guardamos el total de aciertos para cada número
aciertos = [0] * 10
fallos = [0] * 10

for i in range(len(train)):
    prediccion = predict2(train, i, mostrarErrores=False)
    target = int(train.iloc[i]["label"])

    # Sumamos uno si acierta
    if prediccion == target:
        aciertos[target] += 1
    else:
        fallos[target] += 1

In [ ]:
df_resultados2 = calcular_resultados(aciertos, fallos)
df_resultados2

### Método 2B
Binarizamos los números ideales y el número a predecir

In [ ]:
# --- Binarizamos cada número ideal ---
nums_ideales2b = nums_ideales.copy()

for n in range(len(nums_ideales2b)):
    for i in range(nums_ideales2b[n].shape[0]):
        for j in range(nums_ideales2b[n].shape[1]):
            nums_ideales2b[n][i][j] = round(nums_ideales2b[n][i][j])

In [ ]:
mostrarNumsIdeales(nums_ideales2b)

In [ ]:
def predict2b(df, idx, mostrarErrores=True):
    img, label, title = crearMatriz(df, idx)

    # Guardamos la matriz de error de cada número ideal
    matrices_resta = []
    errores = []

    """
    Mismo procedimiento (valor absoluto de la resta), pero primero "binarizamos" la matriz 'img'
    """
    # Valores entre 0 y 1
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            img[i][j] = round(img[i][j])

    for i in range(len(nums_ideales2b)):

        matriz_differencia = np.abs(np.subtract(nums_ideales2b[i], img))
        err = np.sum(matriz_differencia)
        err = float(round(err, 3))  # Para mejorar la lectura de los datos

        matrices_resta.append(matriz_differencia)
        errores.append(err)

    # Normalizamos los errores para conseguir la probabilidad
    # El menor error tendrá la probabilidad mayor (de manera proporcional)
    # Primero, "invertimos" los valores para que el menor error tenga el mayor valor
    max_err = np.max(errores)
    for e in range(len(errores)):
        err = max_err - errores[e]
        errores[e] = float(round(err, 5))

    # Después calculamos la probabilidad normalizada de elegir cada número ideal
    vector_probabilidades = []
    for i in range(len(errores)):
        probabilidad = float(errores[i] / np.sum(errores))
        vector_probabilidades.append(round(probabilidad, 5))

    prediccion = vector_probabilidades.index(max(vector_probabilidades))

    if mostrarErrores:
        print(f"Errores:\n{errores}\tMínimo err: {min(errores)}")
        print(f"Probabilidades:\n{vector_probabilidades} ----> {prediccion}")
        print("sumatorio probs.", np.sum(vector_probabilidades))
        mostrarNumsIdeales(matrices_resta, figsize=(7, 5))

    # Devolvemos el valor del mínimo error; es decir, el índice de la matriz correspondiente
    return prediccion

In [ ]:
# Probamos a predecir todos los números de 'train'          !!!!!!!!!!!!!!!!!!!!!!!!! NO estamos probando en 'test' !!!!!!!!!
# Guardamos el total de aciertos para cada número
aciertos = [0] * 10
fallos = [0] * 10

for i in range(len(train)):
    prediccion = predict2b(train, i, mostrarErrores=False)
    target = int(train.iloc[i]["label"])

    # Sumamos uno si acierta
    if prediccion == target:
        aciertos[target] += 1
    else:
        fallos[target] += 1

In [ ]:
df_resultados2b = calcular_resultados(aciertos, fallos)
df_resultados2b

### Método 3
Distancia euclídea entre vectores

Transformamos cada matriz ideal 28x28 y cada número a predecir en un vector (1, 784) y calculamos su distancia euclídea

In [ ]:
vectores_ideales = []
for num in nums_ideales:
    vector = num.reshape((1, 784))
    vectores_ideales.append(vector)


def predict3(df, idx, mostrar_distancias=False):
    vector_img, label, title = crearMatriz(df, idx)

    # Vectorizamos la imagen
    vector_img = vector_img.reshape((1, 784))

    distancias = []
    # Calculamos la distancia euclídea (Pitágoras)
    for vect in vectores_ideales:
        sumatorio = np.sum((vect - vector_img) ** 2)
        dist = np.sqrt(sumatorio)
        distancias.append(dist)

    # Mismo procedimiento que en el método 1:
    # Nos interesa el menor valor, asi que invertimos los números para
    # calcular las probabilidades
    valor_maximo = np.max(distancias)
    distancias_invertidas = np.subtract(distancias, valor_maximo)

    sum_distancias = np.sum(distancias_invertidas)
    vector_probabilidades = distancias_invertidas / sum_distancias
    vector_probabilidades = list(vector_probabilidades)

    max_probabilidad = max(vector_probabilidades)
    index = vector_probabilidades.index(max_probabilidad)

    if mostrar_distancias:
        print("Distancias:\n", distancias)
        print("Probabilidad:\n", vector_probabilidades, "-->", index)

    return index


In [ ]:
def descargar_vectores_ideales(vectores_ideales):
    # Diccionario en el que guardamos cada pixel como su propia columna
    pixeles = {}
    # Diccionario en el que guardamos cada diccionario de pixeles
    vectores = {}
    dataframe = pd.DataFrame()
    
    for v in range(len(vectores_ideales)):
        # Nos interesa tratar cada vector (1, 784) como una lista de tamaño 784
        # Transponemos el vector (784, 1) y lo transformamos en una lista
        vector = list(vectores_ideales[v].T)
        for i in range(len(vector)):
            pixeles[f"pixel{i}"] = vector[i][0]

        dataframe[f"Vector{v}"] = pixeles
    
    # Transponemos el dataframe
    dataframe = dataframe.transpose()
    dataframe.to_csv("vectores_ideales.csv")
    
# descargar_vectores_ideales(vectores_ideales)

In [ ]:
# --- Evaluación del error "casera" ---
# Probamos a predecir todos los números de 'train'          !!!!!!!!!!!!!!!!!! NO estamos probando en 'test' !!!!!!!!!
# Guardamos el total de aciertos para cada número
aciertos = [0] * 10
fallos = [0] * 10



for i in range(len(train)):
    prediccion = predict3(train, i, mostrar_distancias=False)
    target = int(train.iloc[i]["label"])

    # Sumamos uno si acierta
    if prediccion == target:
        aciertos[target] += 1
    else:
        fallos[target] += 1

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

predictions = []
for i in range(len(X_test)):
    pred = predict3(X_test, i, mostrar_distancias=False)
    predictions.append(pred)


Matriz_de_confusion = confusion_matrix(Y_test, predictions)

Matriz_de_confusion

In [ ]:
precision = precision_score(
    Y_test, predictions, average="macro"
)
recall = recall_score(Y_test, predictions, average="macro")
F1 = f1_score(Y_test, predictions, average="macro")
print("Precisión:", precision)
recall, F1

In [ ]:
df_resultados3 = calcular_resultados(aciertos, fallos)
df_resultados3

### Comparamos los resultados

In [ ]:
print(df_resultados1)           # Diferencia entre matrices                                                     0.595059
print("\n", df_resultados2)     # Diferencia entre matrices (Valores discretos)                                 0.529522
print("\n", df_resultados2b)    # Diferencia entre matrices (Números ideales e imágenes con valores discretos)  0.660281
print("\n", df_resultados3)     # Distancia euclídea de vectores                                                0.661446

# METODO 5 (externo, SVM)

In [ ]:
from sklearn.svm import SVC # Es un modelo de clasficación lineal, lo que hace que se necesite OvO después

svm_clasificador = SVC(gamma="auto", random_state=42, verbose=True)

svm_clasificador.fit(X_entrenamiento, Y_entrenamiento) # entrenar

In [ ]:
some_digit = X_test.iloc[42]
matriz, label, title = crearMatriz(X_test, 42)

mostrarMatrizImagen(matriz, "Número a predecir")

predictions = svm_clasificador.predict([some_digit]) # predecir un ejemplo
prediction = int(predictions[0])

print("Predicción:", prediction)
print(predictions)

# EVALUACION 

## Matriz de confusión

In [ ]:
from sklearn.model_selection import cross_val_predict

# Y_entrenamiento_predicciones = cross_val_predict(svm_clasificador, X_entrenamiento, Y_entrenamiento, cv=3)

In [ ]:
# Predecimos los datos de test para evaluar el modelo
predictions = svm_clasificador.predict(X_test)

In [ ]:
predictions.shape

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# Matriz_de_confusion = confusion_matrix(Y_test, Y_entrenamiento_predicciones)
Matriz_de_confusion = confusion_matrix(Y_test, predictions)

# precision = precision_score(
#     Y_entrenamiento, Y_entrenamiento_predicciones, average="macro"
# )
# recall = recall_score(Y_entrenamiento, Y_entrenamiento_predicciones, average="macro")
# F1 = f1_score(Y_entrenamiento, Y_entrenamiento_predicciones, average="macro")


# si fuese binario: F1 = Matriz_de_confusion[1, 1] / (Matriz_de_confusion[1, 1] + (Matriz_de_confusion[1, 0] + Matriz_de_confusion[0, 1]) / 2)

In [ ]:
Matriz_de_confusion

## Precisión/Recall Trade-off

accedes al score asociado a un digito random

In [ ]:
y_scores = svm_clasificador.decision_function([some_digit]) 
# accedo al score que asigna el clasificador a ese ejemplo en específico

y_scores

In [ ]:
limite = 0 
y_some_digit_pred = (y_scores > limite)

y_some_digit_pred

si establecemos el limite de decisión en 19000, muy probablemente el digito random se clasifica en negativo

In [ ]:
limite = 19000
y_some_digit_pred = (y_scores > limite)
y_some_digit_pred

cojo todos los scores asociados a cada uno de los valores de trainset

In [ ]:
y_scores = cross_val_predict(svm_clasificador, X_entrenamiento, Y_entrenamiento, cv=5, method="decision_function")

In [ ]:
from sklearn.metrics import precision_recall_curve

precisiones, recalls, limites = precision_recall_curve(Y_entrenamiento, y_scores)

In [ ]:
import matplotlib as plt
import numpy as np

def plot_precision_recall_vs_limite(precisions, recalls, limites):
    plt.plot(limites, precisions[:-1], "b--", label="Precision", linewidth=2)
    plt.plot(limites, recalls[:-1], "g-", label="Recall", linewidth=2)



recall_90_precision = recalls[np.argmax(precisiones >= 0.90)]
limite_90_precision = limites[np.argmax(precisiones >= 0.90)]

# Precisiones VS Recalls

In [ ]:
def plot_precision_vs_recall(precisiones, recalls):
    plt.plot(recalls, precisiones, "b-", linewidth=2)
    plt.xlabel("Recall", fontsize=16)
    plt.ylabel("Precision", fontsize=16)
    plt.axis([0, 1, 0, 1])
    plt.grid(True)

plt.figure(figsize=(8, 6))
plot_precision_vs_recall(precisiones, recalls)

In [ ]:
limite_90_precision = limites[np.argmax(precisiones >= 0.90)]
y_entrenamiento_prediccion_90 = (y_scores >= limite_90_precision)

limite_90_precision

In [ ]:
precision_score(Y_entrenamiento, y_entrenamiento_prediccion_90)
recall_score(Y_entrenamiento, y_entrenamiento_prediccion_90)

precision_score
recall_score